* The Scenario: You are given three separate datasets: "User Demographics", "Track Metadata" (artist, genre, duration), and "Listening History" (who listened to what, and when).
* Combining Datasets: You must use pd.concat() to stitch together listening history from Q1 and Q2. Then, use pd.merge() to join the 'Listening * * History' table with the 'Track Metadata' table so you know the genre of the song played, acting like a SQL left join.
Modifying DataFrames: You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. Use .drop() to clear redundant system ID columns and .rename() poorly named columns.
* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the timestamp.
* Grouping and Aggregation: Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.
* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.


• The Scenario: You are given three separate datasets: "User Demographics", "Track Metadata" (artist, genre, duration), 
and "Listening History" (who listened to what, and when).

In [50]:
import pandas as pd
df1 = pd.read_csv('user_demographics.csv')
df1

,user_id,User Age Group,subscription_tier,sys_user_hash
0,1,45-54,Premium,usr_1392
1,2,55+,Free,usr_2144
2,3,35-44,Premium,usr_1874
3,4,55+,Premium,usr_1703
4,5,55+,Free,usr_3886
...,...,...,...,...
195,196,35-44,Premium,usr_3510
196,197,35-44,Free,usr_4601
197,198,18-24,Premium,usr_5062
198,199,55+,Free,usr_7729


In [51]:
df2 = pd.read_csv('track_metadata.csv')
df2

,track_id,artist_name,Genre,length_in_ms,internal_db_id
0,100,Artist_0,Pop,203042,645977
1,101,Artist_1,Pop,205999,897606
2,102,Artist_2,Electronic,137955,450605
3,103,Artist_3,Hip-Hop,157841,989465
4,104,Artist_4,Hip-Hop,190640,633636
5,105,Artist_5,Electronic,285983,138102
6,106,Artist_6,Jazz,258429,174460
7,107,Artist_7,Hip-Hop,283551,464778
8,108,Artist_8,Pop,181476,189930
9,109,Artist_9,Pop,164811,631089


In [52]:
df3 = pd.read_csv('listening_history_q1.csv')
df3

,listen_id,usr_ref_id,trk_ref_id,timestamp
0,1,167,146,2025-01-23 22:19:45
1,3,90,105,2025-01-06 15:32:21
2,5,39,119,2025-02-12 11:12:04
3,6,126,149,2025-03-05 19:52:42
4,8,173,101,2025-01-04 5:56:14
...,...,...,...,...
711,1490,28,127,2025-02-08 9:57:41
712,1491,121,102,2025-03-06 14:50:04
713,1492,107,119,2025-03-08 6:57:40
714,1498,44,102,2025-02-10 11:10:20


In [53]:
df4 = pd.read_csv('listening_history_q2.csv')
df4

,listen_id,usr_ref_id,trk_ref_id,timestamp
0,2,91,114,2025-04-24 23:17:39
1,4,19,143,2025-05-26 1:07:58
2,7,195,113,2025-04-21 10:30:02
3,9,141,108,2025-05-19 3:56:40
4,10,126,136,2025-05-22 1:49:08
...,...,...,...,...
779,1494,175,126,2025-06-15 7:30:32
780,1495,165,101,2025-05-22 13:18:12
781,1496,49,128,2025-06-13 22:29:31
782,1497,120,137,2025-04-26 2:20:18


* Combining Datasets: You must use pd.concat() to stitch together listening history from Q1 and Q2. Then, use pd.merge() to join the 'Listening * * History' table with the 'Track Metadata' table so you know the genre of the song played, acting like a SQL left join. Modifying DataFrames: You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. Use .drop() to clear redundant system ID columns and .rename() poorly named columns.

In [54]:
listening_df = pd.concat([df3, df4], ignore_index=True)
listening_df

,listen_id,usr_ref_id,trk_ref_id,timestamp
0,1,167,146,2025-01-23 22:19:45
1,3,90,105,2025-01-06 15:32:21
2,5,39,119,2025-02-12 11:12:04
3,6,126,149,2025-03-05 19:52:42
4,8,173,101,2025-01-04 5:56:14
...,...,...,...,...
1495,1494,175,126,2025-06-15 7:30:32
1496,1495,165,101,2025-05-22 13:18:12
1497,1496,49,128,2025-06-13 22:29:31
1498,1497,120,137,2025-04-26 2:20:18


In [55]:
merge_df = pd.merge(listening_df, df2, left_on='trk_ref_id', right_on='track_id', how='inner')
merge_df

,listen_id,usr_ref_id,trk_ref_id,timestamp,track_id,artist_name,Genre,length_in_ms,internal_db_id
0,1,167,146,2025-01-23 22:19:45,146,Artist_46,Electronic,121062,119870
1,3,90,105,2025-01-06 15:32:21,105,Artist_5,Electronic,285983,138102
2,5,39,119,2025-02-12 11:12:04,119,Artist_19,Hip-Hop,134397,625830
3,6,126,149,2025-03-05 19:52:42,149,Artist_49,Pop,272617,334677
4,8,173,101,2025-01-04 5:56:14,101,Artist_1,Pop,205999,897606
...,...,...,...,...,...,...,...,...,...
1495,1494,175,126,2025-06-15 7:30:32,126,Artist_26,Rock,298352,319930
1496,1495,165,101,2025-05-22 13:18:12,101,Artist_1,Pop,205999,897606
1497,1496,49,128,2025-06-13 22:29:31,128,Artist_28,Classical,267851,617313
1498,1497,120,137,2025-04-26 2:20:18,137,Artist_37,Hip-Hop,259407,257504


In [56]:
Minutes_played = merge_df['length_in_ms'] / 60000
Minutes_played

0       2.017700
1       4.766383
2       2.239950
3       4.543617
4       3.433317
          ...   
1495    4.972533
1496    3.433317
1497    4.464183
1498    4.323450
1499    4.592300
Name: length_in_ms, Length: 1500, dtype: float64

In [57]:
clear_id = merge_df.drop('internal_db_id', axis=1, errors='ignore')
clear_id

,listen_id,usr_ref_id,trk_ref_id,timestamp,track_id,artist_name,Genre,length_in_ms
0,1,167,146,2025-01-23 22:19:45,146,Artist_46,Electronic,121062
1,3,90,105,2025-01-06 15:32:21,105,Artist_5,Electronic,285983
2,5,39,119,2025-02-12 11:12:04,119,Artist_19,Hip-Hop,134397
3,6,126,149,2025-03-05 19:52:42,149,Artist_49,Pop,272617
4,8,173,101,2025-01-04 5:56:14,101,Artist_1,Pop,205999
...,...,...,...,...,...,...,...,...
1495,1494,175,126,2025-06-15 7:30:32,126,Artist_26,Rock,298352
1496,1495,165,101,2025-05-22 13:18:12,101,Artist_1,Pop,205999
1497,1496,49,128,2025-06-13 22:29:31,128,Artist_28,Classical,267851
1498,1497,120,137,2025-04-26 2:20:18,137,Artist_37,Hip-Hop,259407


In [59]:
merge_df = clear_id.rename(columns={'trk_ref_id':'track_ref_id'})
merge_df

,listen_id,usr_ref_id,track_ref_id,timestamp,track_id,artist_name,Genre,length_in_ms
0,1,167,146,2025-01-23 22:19:45,146,Artist_46,Electronic,121062
1,3,90,105,2025-01-06 15:32:21,105,Artist_5,Electronic,285983
2,5,39,119,2025-02-12 11:12:04,119,Artist_19,Hip-Hop,134397
3,6,126,149,2025-03-05 19:52:42,149,Artist_49,Pop,272617
4,8,173,101,2025-01-04 5:56:14,101,Artist_1,Pop,205999
...,...,...,...,...,...,...,...,...
1495,1494,175,126,2025-06-15 7:30:32,126,Artist_26,Rock,298352
1496,1495,165,101,2025-05-22 13:18:12,101,Artist_1,Pop,205999
1497,1496,49,128,2025-06-13 22:29:31,128,Artist_28,Classical,267851
1498,1497,120,137,2025-04-26 2:20:18,137,Artist_37,Hip-Hop,259407


* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the timestamp.

In [68]:
merge_df['timestamp'] = pd.to_datetime(merge_df['timestamp'])
merge_df['time_of_day'] = merge_df['timestamp'].dt.hour.apply(
    lambda x: 'Morning' if 5 <= x < 12 
    else 'Afternoon' if 12 <= x < 17 
    else 'Evening' if 17 <= x < 21 
    else 'Night'
)
merge_df

,listen_id,usr_ref_id,track_ref_id,timestamp,track_id,artist_name,Genre,length_in_ms,time_of_day
0,1,167,146,2025-01-23 22:19:45,146,Artist_46,Electronic,121062,Night
1,3,90,105,2025-01-06 15:32:21,105,Artist_5,Electronic,285983,Afternoon
2,5,39,119,2025-02-12 11:12:04,119,Artist_19,Hip-Hop,134397,Morning
3,6,126,149,2025-03-05 19:52:42,149,Artist_49,Pop,272617,Evening
4,8,173,101,2025-01-04 05:56:14,101,Artist_1,Pop,205999,Morning
...,...,...,...,...,...,...,...,...,...
1495,1494,175,126,2025-06-15 07:30:32,126,Artist_26,Rock,298352,Morning
1496,1495,165,101,2025-05-22 13:18:12,101,Artist_1,Pop,205999,Afternoon
1497,1496,49,128,2025-06-13 22:29:31,128,Artist_28,Classical,267851,Night
1498,1497,120,137,2025-04-26 02:20:18,137,Artist_37,Hip-Hop,259407,Night


* Grouping and Aggregation: Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.

In [78]:
grouped = merge_df.groupby('Genre').agg({
    'length_in_ms': ['sum', 'mean'],
    'usr_ref_id': 'nunique'
    })
grouped.columns = ['Total_duration', 'Average_duration', 'Unique_listener']
grouped['Total_minutes_played'] = grouped['Total_duration'] / 60000
grouped['Total_average_played'] = grouped['Average_duration'] / 60000
grouped

,Total_duration,Average_duration,Unique_listener,Total_minutes_played,Total_average_played
Genre,,,,,
Classical,12962027,270042.229167,43,216.033783,4.500704
Electronic,79210810,218211.597796,164,1320.180167,3.636860
Hip-Hop,65350078,229298.519298,154,1089.167967,3.821642
Jazz,34166787,204591.538922,117,569.446450,3.409859
Pop,64158361,222772.086806,162,1069.306017,3.712868
Rock,79073209,226570.799427,170,1317.886817,3.776180


* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.

In [89]:
import pandas as pd
import numpy as np
unique_users = merge_df['usr_ref_id'].unique()
user_df = pd.DataFrame({
    'usr_ref_id': unique_users,
    'age': np.random.randint(15, 65, size=len(unique_users)) 
})
bins = [0, 18, 25, 35, 50, 100]
labels = ['<18', '18-24', '25-34', '35-49', '50+']
merge_df = pd.merge(merge_df, user_df[['usr_ref_id', 'age']], on='usr_ref_id', how='left')
merge_df['age_group'] = pd.cut(merge_df['age'], bins=bins, labels=labels, right=False)
heatmap_data = pd.pivot_table(
    merge_df,
    values = 'listen_id',
    index = 'age_group',
    columns = 'Genre',
    aggfunc = 'count'
)
heatmap_data

MergeError: Passing 'suffixes' which cause duplicate columns {'age_x'} is not allowed.